# M4P1 - Introduction to Differential Expression Analysis

In this notebook, we will perform DE analysis on the data set [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275). This is the same data set we had used before during the pseudo-alignment, except this time we will be performing the analysis across the root (OPT3 vs Col-0 and Cd-treated vs untreated replete) samples. The table below shows the non-treated (replete) samples within the dataset, and their conditions: 

| Sample ID   | Replicate Name | Genotype | Tissue | Biological Rep | Technical Rep |
|------------|----------------|----------|--------|----------------|---------------|
| GSM2090459 | CRN1-1         | Col-0    | Root   | 1              | 1             |
| GSM2090460 | CRN1-2         | Col-0    | Root   | 1              | 2             |
| GSM2090461 | CRN1-3         | Col-0    | Root   | 1              | 3             |
| GSM2090462 | CRN2-1         | Col-0    | Root   | 2              | 1             |
| GSM2090463 | CRN2-2         | Col-0    | Root   | 2              | 2             |
| GSM2090464 | CRN2-3         | Col-0    | Root   | 2              | 3             |
| GSM2090465 | CRN3-1         | Col-0    | Root   | 3              | 1             |
| GSM2090466 | CRN3-2         | Col-0    | Root   | 3              | 2             |
| GSM2090467 | CRN3-3         | Col-0    | Root   | 3              | 3             |
| GSM2090468 | CLN1-1         | Col-0    | Leaf   | 1              | 1             |
| GSM2090469 | CLN1-2         | Col-0    | Leaf   | 1              | 2             |
| GSM2090470 | CLN1-3         | Col-0    | Leaf   | 1              | 3             |
| GSM2090471 | CLN2-1         | Col-0    | Leaf   | 2              | 1             |
| GSM2090472 | CLN2-2         | Col-0    | Leaf   | 2              | 2             |
| GSM2090473 | CLN2-3         | Col-0    | Leaf   | 2              | 3             |
| GSM2090474 | CLN3-1         | Col-0    | Leaf   | 3              | 1             |
| GSM2090475 | CLN3-2         | Col-0    | Leaf   | 3              | 2             |
| GSM2090476 | CLN3-3         | Col-0    | Leaf   | 3              | 3             |
| GSM2090477 | ORN1-1         | opt3-2   | Root   | 1              | 1             |
| GSM2090478 | ORN1-2         | opt3-2   | Root   | 1              | 2             |
| GSM2090479 | ORN1-3         | opt3-2   | Root   | 1              | 3             |
| GSM2090480 | ORN2-1         | opt3-2   | Root   | 2              | 1             |
| GSM2090481 | ORN2-2         | opt3-2   | Root   | 2              | 2             |
| GSM2090482 | ORN2-3         | opt3-2   | Root   | 2              | 3             |
| GSM2090483 | ORN3-1         | opt3-2   | Root   | 3              | 1             |
| GSM2090484 | ORN3-2         | opt3-2   | Root   | 3              | 2             |
| GSM2090485 | ORN3-3         | opt3-2   | Root   | 3              | 3             |
| GSM2090486 | OLN1-1         | opt3-2   | Leaf   | 1              | 1             |
| GSM2090487 | OLN1-2         | opt3-2   | Leaf   | 1              | 2             |
| GSM2090488 | OLN1-3         | opt3-2   | Leaf   | 1              | 3             |
| GSM2090489 | OLN2-1         | opt3-2   | Leaf   | 2              | 1             |
| GSM2090490 | OLN2-2         | opt3-2   | Leaf   | 2              | 2             |
| GSM2090491 | OLN2-3         | opt3-2   | Leaf   | 2              | 3             |
| GSM2090492 | OLN3-1         | opt3-2   | Leaf   | 3              | 1             |
| GSM2090493 | OLN3-2         | opt3-2   | Leaf   | 3              | 2             |
| GSM2090494 | OLN3-3         | opt3-2   | Leaf   | 3              | 3             |

**According to the [Metadata](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE79nnn/GSE79275/miniml/), this is how the data was processed:**
- Read Cleaning: Reads were cleaned in R using ShortRead::TrimEnds, removing bases from each end until the remaining read has bases with p-values all over 0.1.  Afterwards, reads shorter than 25bp were removed.
- Read Alignment: Reads were aligned to the TAIR10 genome using Tophat2 using default parameters except -g 1, specifying to only retain reads with one valid alignment
- Read counting: Reads were counted in R using GenomicAlignments::summarizeOverlaps with the TAIR10 gff as input and the counting mode of 'union'
- FPKM calculations: pseudo-counts (generated from edgeR's library size normalization) were divided by the gene length (in kb) found in the TAIR 10 gff.

This differs from our pseudo-alignment with `salmon` last week, but the concept is the same. Samples were aligned to a map, and reads were counted. 

In this notebook we follow these steps to complete DE Analysis using `DESeq2`:
1. [Setup](#1)
2. [Exploring RNA-seq count data](#2)
3. [Count Normalization](#3)
4. [Quality Control on Counts](#4)
5. [Differential expression analysis with DESeq2](#5)
6. [Wald Test](#6)
7. [Summarizing Results](#7)
8. [Visualizing the results](#8)

```{admonition} Adapted from
:class: tip
**Source:** [Introduction to DGE](https://hbctraining.github.io/Intro-to-DGE/), Harvard Chan Bioinformatics Core (HBC) training team, [hbctraining/Intro-to-DGE](https://github.com/hbctraining/Intro-to-DGE) (DESeq2 lessons: count normalization, sample-level QC, DE analysis, Wald test results).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Figures:** unless a caption says otherwise, the diagrams and example plots in this notebook come from the same HBC lessons, CC BY 4.0, unchanged.

**Changes:** The HBC Mov10 example data were replaced with the Arabidopsis GSE79275 count matrix (opt3-2 and Col-0 roots/shoots, with and without Cd); R code was adapted to this design; questions were added for the course; some third-party figures were removed.
```

## 1. Setup<a id="1"></a>

#### Loading libraries

For this analysis, we will be using several R packages, some of which have been installed from CRAN and others from Bioconductor. Make sure we are using the Module 3 R kernel. To use these packages (and the functions contained within them), we need to load the libraries. Add the following to your script and don’t forget to comment liberally!

In [ ]:
### Bioconductor and CRAN libraries used
library(DESeq2)
library(tidyverse)
library(RColorBrewer)
library(pheatmap)
library(tximport)
library(ggplot2)
library(ggrepel)

### Loading data

The data we are using is downloaded to `data/geo/GSE79275/GSE79275_raw_read_counts.txt` by `data/fetch_data.sh geo`. This contains the raw reads for the experiment within [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275). 

Of great importance is keeping track of the information about our data. At a minimum, we need a metadata data frame that maps our samples to the experimental groups we are investigating. This allows DESeq2 (or any other RNA-Seq analysis tool) to understand which samples belong to which conditions, tissues, genotypes, and replicates.

In our case, the metadata table has:

- Row names: the column headers from the counts matrix (the BAM files or sample names).

- Columns: descriptive information about each sample.

In [ ]:
# Import the counts table
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder; run data/fetch_data.sh geo first
counts <- read.delim(file.path(DATA_DIR, "geo", "GSE79275", "GSE79275_raw_read_counts.txt"), header = TRUE, stringsAsFactors = FALSE)

# Make rownames the gene IDs (first column)
rownames(counts) <- counts$AGI
counts <- counts[ , -1]  # remove first column after setting rownames

# Ensure all counts are integers
counts <- round(as.matrix(counts))

# Get sample names from counts table
sample_names <- colnames(counts)

# Initialize vectors
genotype <- character(length(sample_names))
tissue <- character(length(sample_names))
treatment <- character(length(sample_names))
bio_rep <- integer(length(sample_names))
tech_rep <- integer(length(sample_names))

# Parse sample names
for (i in seq_along(sample_names)) {
  s <- sample_names[i]
  
  # Genotype
  if (grepl("^C", s)) {
    genotype[i] <- "col0"
  } else if (grepl("^O", s)) {
    genotype[i] <- "opt3"
  }
  
  # Tissue
  if (grepl("R", s)) {
    tissue[i] <- "Root"
  } else if (grepl("S", s)) {
    tissue[i] <- "Shoot"
  }
  
  # Treatment
  if (grepl("T", s)) {
    treatment[i] <- "Treated"
  } else {
    treatment[i] <- "Untreated"
  }
  
  # Biological replicate
  bio_rep[i] <- as.integer(sub(".*?([0-9]+)[-_][0-9]+\\.bam$", "\\1", s))
}

# Assign technical replicates sequentially for each bio_rep
colData <- data.frame(
  sample = sample_names,
  genotype = factor(genotype),
  tissue = factor(tissue),
  treatment = factor(treatment),
  bio_rep = bio_rep,
  stringsAsFactors = FALSE
)

# Order by bio_rep and assign tech_rep
colData <- colData[order(colData$bio_rep, colData$sample), ]
colData$tech_rep <- ave(colData$bio_rep, colData$bio_rep, FUN = seq_along)

# Set row names to sample names for DESeq2
rownames(colData) <- colData$sample
colData$sample <- NULL

# Inspect
colData

# Move to appropriate naming
data <- counts
meta <- colData

In [ ]:
#check inputs
data
meta

---
## 2. Exploring RNA-seq count data<a id="2"></a>
#### Count matrix

When we start our differential gene expression analysis, we begin with a matrix summarizing the gene-level expression in each sample of your dataset. The rows in the matrix correspond to genes, and the columns correspond to samples. In each position of the matrix, you will have an integer value representing the total number of sequence reads that originated from a particular gene in a sample.

![deseq_counts_overview.png](images/M4P1_Intro_to_DE_Analysis_9b5081a6.png)

The higher the number of counts, indicates more reads are associated with that gene and suggest a higher level of expression of that gene. However, this is not necessarily true, and we will delve deeper into this later in this lesson and in the course.

#### Characteristics of RNA-seq count data

To get an idea about how RNA-seq counts are distributed, let’s plot a histogram of the counts for a single sample, ‘CR1_5.bam’:

In [ ]:
ggplot(data) +
  geom_histogram(aes(x = CR1_5.bam), stat = "bin", bins = 200) +
  xlab("Raw expression counts") +
  ylab("Number of genes")

This plot illustrates some common features of RNA-seq count data:

- a low number of counts associated with a large proportion of genes
- a long right tail due to the lack of any upper limit for expression
- large dynamic range

Looking at the shape of the histogram, we see that it is not normally distributed. For RNA-seq data, this will always be the case. Moreover, the underlying data, as we observed earlier, are integer counts rather than continuous measurements. We need to take these characteristics into account when deciding on what statistical model to use.

#### Modeling count data

Count data in general can be modeled with various distributions:

- Binomial distribution: Gives you the probability of getting a number of heads upon tossing a coin a number of times. Based on discrete events and used in situations when you have a certain number of cases. Thus, it gives the probability of getting r events out of n. trials

- Poisson distribution: For use when the number of cases is very large (i.e. people who buy lottery tickets), but the probability of an event is very small (probability of winning). The Poisson is similar to the binomial as it is also based on discrete events, but used with data from an infinite sample. Thus, it gives the probability of getting r events in a population. It is appropriate for data where mean == variance.

**So what do we use for RNA-seq count data?**

With RNA-Seq data, a very large number of RNAs are represented, and the probability of pulling out a particular transcript is very small. This scenario is most similar to the lottery described above, suggesting that perhaps the Poisson distribution is most appropriate. However, this will depend on the relationship between mean and variance in our data.

#### Mean versus variance

To assess the properties of the data we are working with, we can use the samples corresponding to the ‘CS1’, or Col-0 Shoot Untreated Sample 1, technical replicates. First, compute a vector of mean values, then compute a vector of variance values. Then plot these values against each other to evaluate the relationship between them.

In [ ]:
mean_counts <- apply(data[,6:8], 1, mean)        #The second argument '1' of 'apply' function indicates the function being applied to rows. Use '2' if applied to columns 
variance_counts <- apply(data[,6:8], 1, var)
df <- data.frame(mean_counts, variance_counts)

ggplot(df) +
        geom_point(aes(x=mean_counts, y=variance_counts)) + 
        scale_y_log10(limits = c(1,1e9)) +
        scale_x_log10(limits = c(1,1e9)) +
        geom_abline(intercept = 0, slope = 1, color="red")

We can see in the plot above: 
1. The mean is not equal to the variance (the scatter of data points does not fall on the diagonal).
2. For the genes with high mean expression, the variance across replicates tends to be greater than the mean (scatter is above the red line).
3.  For the genes with low mean expression, we see quite a bit of scatter. We usually refer to this as “heteroscedasticity”. That is, for a given expression level in the low range we observe a lot of variability in the variance values.


#### An alternative: The Negative Binomial distribution

Our data fail to satisfy the criteria for the Poisson distribution, and typical RNA-seq data will do the same. If the proportions of mRNA stayed exactly constant between the biological replicates for a sample group, we could expect a Poisson distribution (where mean == variance). However, we always expect some amount of variability between replicates (we’ll discuss this in more detail later in the lesson). Alternatively, if we continued to add more replicates (i.e.> 20), we should eventually see the scatter start to reduce and the high expression data points move closer to the red line. So in theory, if we had enough replicates, we could use the Poisson.

In practice, a large number of replicates can be either hard to obtain (depending on how samples are obtained) and/or unaffordable. It is more common to see datasets with only a handful of replicates (~3-5) and a reasonable amount of variation between them. The distribution that fits RNA-seq data best, given this type of variability between replicates, is the Negative Binomial. Essentially, the Negative Binomial is a good approximation for data where the mean < variance, as is the case with RNA-Seq count data.

*NOTE: If we use the Poisson, this will underestimate variability,y leading to an increase in false positive DE genes.*

![deseq_nb.png](images/M4P1_Intro_to_DE_Analysis_bc28d17e.png)

## Questions: 

1. An RNA-seq experiment was conducted on Arabidopsis plants to evaluate the effect of increasing concentrations of a nutrient treatment. For each of the five different concentrations, we have n = 5 plants for a total of 25 samples. If we observed little to no variability between replicates, what might this suggest about our samples?
2. What type of mean-variance relationship would you expect to see for this dataset?


### Replicates and variability

Biological replicates represent multiple samples (i.e. RNA from different plants) that correspond to the same sample class or group. Intuitively, we would expect samples from the same sample group (i.e. that under similar conditions/perturbations) to exhibit a similar transcriptional profile. In most cases, there will be a high degree of similarity of samples within a group, but there will also inevitably be many differences. The source of this variability on a gene’s expression can be attributed to many factors, some of which are identifiable and others that remain unknown.

*[Figure omitted (image courtesy of Paul Pavlidis, UBC, as reproduced in the HBC lessons).]*

With differential expression analysis, we are looking for genes that change in expression between two or more groups. For example,

- case vs. control
- correlation of expression with some variable or clinical outcome

However, there is much more going on with your data than what you are anticipating. Genes that vary in expression level is a consequence of not only the experimental variable(s) of interest but also due to extraneous sources. The goal of differential expression analysis is to identify and correct for sources of variation such that we can separate the “interesting” from the “uninteresting”.

Let’s take a closer look at the figure below as an example. Expression (counts) is plotted here for ‘GeneA’ in the ‘untreated’ and ‘treated’ groups. Each dot corresponds to an expression for a single sample, and the dots are colored based on which group they belong to.

![de_norm_counts_var.png](images/M4P1_Intro_to_DE_Analysis_784a5183.png)

The mean expression level of GeneA for the ‘treated’ group is twice as large as the mean expression level for the ‘untreated’ group. But is the difference in expression between groups significant, given the amount of variation observed within groups (across replicates)?

It is possible that the difference is not actually significant. We need to take into account the variation in the data (and where it might be coming from) when determining whether genes are differentially expressed. Modeling our data with the negative binomial distribution allows us to do this.

---
# Differential Expression with DESeq2

`DESeq2` is a popular tool for gene-level differential expression analysis. It uses the negative binomial distribution, employing a slightly more stringent approach compared to some methods, yet having a good balance between sensitivity and specificity (reducing both false positives and false negatives).

We will be using `DESeq2` for the analysis in this practice. The analysis steps with `DESeq2` are shown in the flowchart below in green and blue.

![de_workflow_salmon.png](images/M4P1_Intro_to_DE_Analysis_1d3e7e73.png)

*NOTE: Remember, these counts were aligned using `Tophat2`, not a pseudo-aligner, but the concept is the same.*


We will go in-depth into each of these steps in the following cells, but additional details and helpful suggestions regarding DESeq2 can be found in the [DESeq2 vignette](http://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html).

# 3. Count Normalization <a id="3"></a>

The first step in the DE analysis workflow is count normalization, which is necessary to make accurate comparisons of gene expression between samples.

![de_workflow_salmon_normalization.png](images/M4P1_Intro_to_DE_Analysis_da1231a0.png)

The counts of mapped reads for each gene is the result of RNA expression (“interesting” part) in addition to many other factors (“uninteresting” part). Normalization is the process of adjusting raw count values to account for the “uninteresting” factors. In this way, the expression levels are more comparable between and/or within samples.

The main “uninteresting” factors often considered during normalization are:

- Sequencing depth: Accounting for sequencing depth is necessary for comparison of gene expression between samples. In the example below, each gene appears to have twice the expression in Sample A relative to Sample B. However, this is the consequence of Sample A having doubled sequencing depth.

![normalization_methods_depth.png](images/M4P1_Intro_to_DE_Analysis_4342ad58.png)

*NOTE: In the figure above, each pink and green rectangle represents a read aligned to a gene. Reads connected by dashed lines connect a read spanning an intron.*

- Gene length: Accounting for gene length is necessary for comparing expression between different genes within the same sample. In the example below, Gene X and Gene Y have similar levels of expression, but the number of reads mapped to Gene X would be many more than the number mapped to Gene Y because Gene X is longer.

![normalization_methods_length.png](images/M4P1_Intro_to_DE_Analysis_75096fcd.png)

- RNA composition: A few highly differentially expressed genes between samples, differences in the number of genes expressed between samples, or the presence of contamination can skew some types of normalization methods. Accounting for RNA composition is recommended for accurate comparison of expression between samples, and is particularly important when performing differential expression analyses [1].

    In the example below, imagine the sequencing depths are similar between Sample A and Sample B, and every gene except for gene DE presents a similar expression level between samples. The counts in Sample B would be greatly skewed by the DE gene, which takes up most of the counts. Other genes for Sample B would therefore appear to be less expressed than those same genes in Sample A.

![normalization_methods_composition.png](images/M4P1_Intro_to_DE_Analysis_044c93df.png)

**While normalization is essential for differential expression analyses, it is also necessary for exploratory data analysis, visualization of data, and whenever you are exploring or comparing counts between or within samples.
Common normalization methods**

Several common normalization methods have been developed to account for these differences:

| Normalization method | Description | Accounted factors | Recommendations for use |
|-----------------------|-------------|-------------------|--------------------------|
| **CPM (counts per million)** | Counts scaled by total number of reads | Sequencing depth | Gene count comparisons **between replicates of the same sample group**; **NOT** for within-sample comparisons or DE analysis |
| **TPM (transcripts per kilobase million)** | Counts per length of transcript (kb) per million reads mapped | Sequencing depth and gene length | Gene count comparisons **within a sample** or **between samples of the same sample group**; **NOT** for DE analysis |
| **RPKM/FPKM (reads/fragments per kilobase of exon per million reads/fragments mapped)** | Similar to TPM | Sequencing depth and gene length | Gene count comparisons **between genes within a sample**; **NOT** for between-sample comparisons or DE analysis |
| **DESeq2’s median of ratios** | Counts divided by sample-specific size factors determined by median ratio of gene counts relative to geometric mean per gene | Sequencing depth and RNA composition | Gene count comparisons **between samples** and for **DE analysis**; **NOT** for within-sample comparisons |
| **EdgeR’s trimmed mean of M values (TMM)** | Uses a weighted trimmed mean of the log expression ratios between samples | Sequencing depth and RNA composition | Gene count comparisons **between samples** and for **DE analysis**; **NOT** for within-sample comparisons |
***RPKM/FPKM:* not recommended for between-sample comparisons**

While TPM and RPKM/FPKM normalization methods both account for sequencing depth and gene length, RPKM/FPKM are not recommended. **The reason is that the normalized count values output by the RPKM/FPKM method are not comparable between samples.**

Using RPKM/FPKM normalization, the total number of RPKM/FPKM normalized counts for each sample will be different. Therefore, you cannot compare the normalized counts for each gene equally between samples.


### DESeq2-normalized counts: Median of ratios method

Since tools for differential expression analysis are comparing the counts of the same gene between sample groups, gene length does not need to be accounted for by the tool. However, sequencing depth and RNA composition do need to be taken into account.

To normalize for sequencing depth and RNA composition, DESeq2 uses the median of ratios method. On the user-end there is only one step, but on the back-end, there are multiple steps involved, as described below.

    NOTE: The steps below describe in detail some of the steps performed by DESeq2 when you run a single function to get DE genes. Basically, for a typical RNA-seq analysis, you would not run these steps individually.

**Step 1:** creates a pseudo-reference sample (row-wise geometric mean)

For each gene, a pseudo-reference sample is created that is equal to the geometric mean across all samples.

![image.png](images/M4P1_Intro_to_DE_Analysis_ce288148.png)

**Step 2:** calculates the ratio of each sample to the reference
For every gene in every sample, the ratios (sample/ref) are calculated (as shown below). Since the majority of genes are not differentially expressed, the majority of genes in each sample should have similar ratios within the sample.

![image.png](images/M4P1_Intro_to_DE_Analysis_45755f02.png)

**Step 3:** calculate the normalization factor for each sample (size factor)

The median value (column-wise for the above table) of all ratios for a given sample is taken as the normalization factor (size factor) for that sample, as calculated below. Notice that the differentially expressed genes should not affect the median value:
```
normalization_factor_sampleA <- median(c(1.28, 1.3, 1.39, 1.35, 0.59))

normalization_factor_sampleB <- median(c(0.78, 0.77, 0.72, 0.74, 1.35))
```
The figure below illustrates the median value for the distribution of all gene ratios for a single sample (y-axis is the frequency).

![deseq_median_of_ratios.png](images/M4P1_Intro_to_DE_Analysis_54969c76.png)

The median of ratios method makes the assumption that not ALL genes are differentially expressed; therefore, the normalization factors should account for sequencing depth and RNA composition of the sample (large outlier genes will not affect the median ratio values). This method is robust to imbalance in up-/down-regulation and large numbers of differentially expressed genes.

**Step 4:** Calculate the normalized count values using the normalization factor

This is performed by dividing each raw count value in a given sample by that sample’s normalization factor, generating normalized count values. This is performed for all count values (every gene in every sample). For example, if the median ratio for sampleA was 1.3 and the median ratio for sampleB was 0.77, you could calculate normalized counts as follows:

![image.png](images/M4P1_Intro_to_DE_Analysis_d98becb8.png)

#### Count normalization of GSE79275 dataset using DESeq2

Now that we know the theory of count normalization, we will normalize the counts for the GSE79275 dataset using DESeq2. This requires a few steps:

- a. Ensure that the row names of the metadata dataframe are present and are in the same order as the column names of the counts dataframe.
- b. Combine our technical replicates
- c. Isolate only root samples
- d. Create a DESeqDataSet object
- e. Generate the normalized counts

**1. Match the metadata and counts data**

We should always make sure that we have sample names that match between the two files, and that the samples are in the same order. DESeq2 will output an error if this is not the case.

In [ ]:
### Check that sample names match in both files
all(colnames(data) %in% rownames(meta))
all(colnames(data) == rownames(meta))

In [ ]:
# Reorder metadata rows to match the count matrix
meta <- meta[colnames(data), ]
all(colnames(data) %in% rownames(meta))
all(colnames(data) == rownames(meta))

In [ ]:
#Combine technical replicates

# Add a column identifying unique biological samples
meta$bio_id <- paste(meta$genotype, meta$tissue, meta$treatment, meta$bio_rep, sep="_")

# Sum counts for technical replicates
collapsed_counts <- sapply(unique(meta$bio_id), function(id) {
  rowSums(data[, meta$bio_id == id])
})

# Make it a proper matrix
collapsed_counts <- as.matrix(collapsed_counts)

# Update metadata to include only biological replicates
collapsed_meta <- meta %>%
  distinct(bio_id, .keep_all = TRUE) %>%
  dplyr::select(-tech_rep)
rownames(collapsed_meta) <- collapsed_meta$bio_id

In [ ]:
collapsed_meta

In [ ]:
#Now we subset for only roots: 
# Subset metadata for root samples only
root_meta <- collapsed_meta[collapsed_meta$tissue == "Root", ]

# Subset counts matrix to match only the root samples
root_counts <- collapsed_counts[, rownames(root_meta)]

# Optional: check dimensions
dim(root_counts)
dim(root_meta)

In [ ]:
#Now we reference our level factors: 
# Ensure Col-0 is the reference for genotype
root_meta$genotype <- relevel(root_meta$genotype, ref = "col0")

# Ensure Untreated is the reference for treatment
root_meta$treatment <- relevel(root_meta$treatment, ref = "Untreated")

**2. Create DESEq2 object**

Bioconductor software packages often define and use custom data structures within R for storing input data, intermediate results, and analysis outputs. These structures are similar to lists in that they can contain multiple types of data, but they differ in that they have pre-defined “slots” designed to hold specific types of information. Access to the data in these slots is provided through package-specific functions.

In this step, we will create a DESeqDataSet object, which is the central data container used by the DESeq2 package. To build this object, we will use our raw count matrix and metadata table as inputs. The count matrix contains the gene expression values (with genes as rows and samples as columns), while the metadata table describes each sample’s experimental condition (such as genotype and treatment).

We must also specify a design formula, which defines the experimental factors we want to test for differential expression. For our dataset, we are interested in modeling gene expression changes with respect to genotype and treatment and their interaction(`genotype:treatment), so our design formula will be:

```r
design = ~ genotype + treatment + genotype:treatment
```
Since our data are already in a gene-level count matrix, we can create the `DESeq2` object directly using the `DESeqDataSetFromMatrix()` function, rather than importing counts through `tximport`. This function will use the count and metadata information to initialize a `DESeqDataSet`, or `dds`, ready for downstream normalization and differential expression analysis.


In [ ]:
dds <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)

You can use DESeq-specific functions to access the different slots and retrieve information. For example, suppose we want to retrieve the original count matrix, we would use `counts()` function (Note: we nest it within the `View()` function so that we can view the result in the script editor rather than in the console) :

In [ ]:
View(counts(dds))

**3. Generate the GSE79275 normalized counts**

The next step is to normalize the count data to make fair gene comparisons between samples.

To perform the median of ratios method of normalization, DESeq2 has a single `estimateSizeFactors()` function that will generate size factors. We will demonstrate this function in the example below, but in a typical RNA-seq analysis, this step is automatically performed by the `DESeq()` function, which we will discuss later.

In [ ]:
dds <- estimateSizeFactors(dds)

By assigning the results back to the `dds` object, we are filling in the slots of the DESeqDataSet object with the appropriate information. We can take a look at the normalization factors of each sample using:

In [ ]:
sizeFactors(dds)

Now, to retrieve the normalized counts matrix from dds, we use the `counts()` function and add the argument `normalized=TRUE`.

In [ ]:
normalized_counts <- counts(dds, normalized=TRUE)

We can save this normalized data matrix to file for later use:

In [ ]:
dir.create("results", showWarnings = FALSE)
write.table(normalized_counts, file="results/GSE79275_normalized_counts.txt", sep="\t", quote=F, col.names=NA)

*NOTE: DESeq2 doesn’t actually use normalized counts; rather, it uses the raw counts and models the normalization inside the Generalized Linear Model (GLM). These normalized counts will be useful for downstream visualization of results, but cannot be used as input to DESeq2 or any other tools that perform differential expression analysis that use the negative binomial model.*

---

# 4. Quality Control on Counts<a id="4"></a>

The next step in the DESeq2 workflow is QC, which includes sample-level and gene-level steps to perform QC checks on the count data to help us ensure that the samples/replicates look good.

![de_workflow_salmon_qc.png](images/M4P1_Intro_to_DE_Analysis_5625f5c0.png)

### Sample-level QC

A useful initial step in an RNA-seq analysis is often to assess overall similarity between samples:

- Which samples are similar to each other, which are different?
- Does this fit the expectation from the experiment’s design?
- What are the major sources of variation in the dataset?

To explore the similarity of our samples, we will be performing sample-level QC using Principal Component Analysis (PCA) and hierarchical clustering methods. These methods/tools allow us to check how well similar the replicates are to each other (clustering) and to make sure that the experimental condition is the major source of variation in the data. Sample-level QC can also help identify any samples behaving like outliers; we can further explore any potential outliers to determine whether they need to be removed before DE analysis.

![sample_qc.png](images/M4P1_Intro_to_DE_Analysis_7074a502.png)

These unsupervised clustering methods are run using log2-transformed normalized counts. The log2 transformation improves the distances/clustering for visualization. Instead of using an ordinary log2 transform, we will be using a regularized log transform (rlog), to avoid any bias from the abundance of low-count genes; Note 1 below explains this in more detail.

![rlog_transformation_new.png](images/M4P1_Intro_to_DE_Analysis_449c0d0e.png)

*Image: from the DESeq2 documentation (Love, Huber & Anders), as reproduced in the HBC Intro-to-DGE lessons (CC BY 4.0).*

### Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a technique used to emphasize variation and bring out strong patterns in a dataset (dimensionality reduction). This is a very important technique used in the QC and analysis of both bulk and single-cell RNAseq data.

To better understand how it works, please go through [this YouTube video from StatQuest that explains PCA](https://www.youtube.com/watch?v=_UVHneBUBW0). After you have gone through the material, please proceed with the interpretation section below.

#### Interpreting PCA plots

Essentially, if two samples have similar levels of expression for the genes that contribute significantly to the variation represented by a given PC (Principal Component), they will be plotted close together on the axis that represents that PC. Therefore, we would expect that biological replicates to have similar scores (because our expectation is that the same genes are changing) and cluster together. This is easiest to understand by visualizing some example PCA plots.

We have an example dataset and a few associated PCA plots below to get a feel for how to interpret them. The metadata for the experiment is displayed below. The main condition of interest is treatment.
![image.png](images/M4P1_Intro_to_DE_Analysis_3b7c2ff6.png)

When visualizing on PC1 and PC2, we don’t see the samples separate by treatment, so we decide to explore other sources of variation present in the data. We hope that we have included all possible known sources of variation in our metadata table, and we can use these factors to color the PCA plot.
![example_PCA_treatmentPC1.png](images/M4P1_Intro_to_DE_Analysis_057616e1.png)

We start with the factor `cage`, but the `cage` factor does not seem to explain the variation on PC1 or PC2.

![example_PCA_cage.png](images/M4P1_Intro_to_DE_Analysis_aa0a6160.png)

Then, we color by the `sex` factor, which appears to separate samples on PC2. This is good information to take note of, as we can use it downstream to account for the variation due to `sex` in the model and regress it out.

![example_PCA_sex.png](images/M4P1_Intro_to_DE_Analysis_2d495347.png)

Next, we explore the `strain` factor and find that it explains the variation on PC1.

![example_PCA_strain.png](images/M4P1_Intro_to_DE_Analysis_28d5c240.png)

It’s great that we have been able to identify the sources of variation for both PC1 and PC2. By accounting for it in our model, we should be able to detect more genes differentially expressed due to treatment.

Worrisome about this plot is that we see two samples that do not cluster with the correct strain. This would indicate a likely sample swap and should be investigated to determine whether these samples are indeed the labeled strains. If we found there was a switch, we could swap the samples in the metadata. However, if we think they are labeled correctly or are unsure, we could just remove the samples from the dataset.

Still, we haven’t found whether `treatment` is a major source of variation after `strain` and `sex`. So, we explore PC3 and PC4 to see if `treatment` is driving the variation represented by either of these PCs.

**Depending on how much variation is explained by the first few principal components, you may want to explore more (i.e consider more components and plot pairwise combinations). Even if your samples do not separate clearly by the experimental variable, you may still get biologically relevant results from the DE analysis. If you are expecting very small effect sizes, then it’s possible the signal is drowned out by extraneous sources of variation. In situations where you can identify those sources, it is important to account for these in your model, as it provides more power to the tool for detecting DE genes.**

#### Hierarchical Clustering Heatmap

Similar to PCA, hierarchical clustering is another complementary method for identifying strong patterns in a dataset and potential outliers. The heatmap displays the correlation of gene expression for all pairwise combinations of samples in the dataset. Since the majority of genes are not differentially expressed, samples generally have high correlations with each other (values higher than 0.80). Samples below 0.80 may indicate an outlier in your data and/or sample contamination.

The hierarchical tree along the axes indicates which samples are more similar to each other, i.e., cluster together. The color blocks at the top indicate substructure in the data, and you would expect to see your replicates cluster together as a block for each sample group. Our expectation would be that the samples cluster together, similar to the groupings we’ve observed in the PCA plot.

In the plot below, we would be quite concerned about ‘Wt_3’ and ‘KD_3’ samples not clustering with the other replicates. We would want to explore the PCA to see if we see the same clustering of samples.

![heatmap_example.png](images/M4P1_Intro_to_DE_Analysis_baeeea90.png)

### GSE79275 quality assessment and exploratory analysis using DESeq2

Now that we have a good understanding of the QC steps normally employed for RNA-seq, let’s implement them for the dataset we are going to be working with.
Transform normalized counts for the dataset

To improve the distances/clustering for the PCA and hierarchical clustering visualization methods, we need to moderate the variance across the mean by applying the rlog transformation to the normalized counts.

    The rlog transformation of the normalized counts is only necessary for these visualization methods during this quality assessment. We will not be using these transformed counts for determining differential expression.


In [ ]:
### Transform counts for data visualization
vsd <- vst(dds, blind=TRUE)

The blind=TRUE argument is to make sure that the rlog() function does not take our sample groups into account - i.e. does the transformation in an unbiased manner. When performing quality assessment, it is important to include this option. The DESeq2 vignette has more details about this.

The rlog() function returns a DESeqTransform object, another type of DESeq-specific object. The reason you don’t just get a matrix of transformed values is because all of the parameters (i.e. size factors) that went into computing the rlog transform are stored in that object. We use this object to plot the PCA and heirarchical clustering figures for quality assessment.

    NOTE: The rlog() funtion can be a bit slow when you have e.g. > 20 samples. In these situations the vst() function is much faster and performs a similar transformation appropriate for use with plotPCA(). It’s typically just a few seconds with vst() due to optimizations and the nature of the transformation.

#### Principal component analysis (PCA) for the GSE79275 dataset

We are now ready for the QC steps, let’s start with PCA!

DESeq2 has a built-in function for generating PCA plots using ggplot2 under the hood. This is great because it saves us having to type out lines of code and having to fiddle with the different ggplot2 layers. In addition, it takes the rlog object as an input directly, hence saving us the trouble of extracting the relevant information from it.

The function `plotPCA()` requires two arguments as input: a `DESeqTransform` object and the “intgroup” (interesting group), i.e. the name of the column in our metadata that has information about the experimental sample groups.

In [ ]:
colnames(colData(dds))

In [ ]:
### Plot PCA 
plotPCA(vsd, intgroup="genotype")

In [ ]:
plotPCA(vsd, intgroup="treatment")

## Questions: 

- What does the above plot tell you about the similarity of samples?
- Does it fit the expectation from the experimental design?
- What do you think the %variance information (in the axes titles) tell you about the data in the context of the PCA?


By default plotPCA() uses the top 500 most variable genes. You can change this by adding the ntop= argument and specifying how many of the genes you want the function to consider.

    NOTE: The plotPCA() function will only return the values for PC1 and PC2. If you would like to explore the additional PCs in your data or if you would like to identify genes that contribute most to the PCs, you can use the prcomp() function. For example, to plot any of the PCs we could run the following code:
```r
     # Input is a matrix of log transformed values
     vsd <- vst(dds, blind=T)
     vsd_mat <- assay(vsd)
     pca <- prcomp(t(vsd_mat))

     # Create data frame with metadata and PC3 and PC4 values for input to ggplot
     df <- cbind(meta, pca$x)
     ggplot(df) + geom_point(aes(x=PC3, y=PC4, color = treatment))
```

#### Hierarchical Clustering for the GSE79275 dataset

There is no built-in function in DESeq2 for plotting the heatmap for displaying the pairwise correlation between all the samples and the hierarchical clustering information; we will use the `pheatmap()` function from the pheatmap package. This function cannot use the DESeqTransform object as input, but requires a matrix or dataframe. So, the first thing to do is retrieve that information from the vsd object using a function called `assay()` (from the SummarizedExperiment package) that converts the data in a DESeqTransform object to a simple 2-dimensional data structure (a matrix in this case).

In [ ]:
### Extract the vst matrix from the object
vsd_mat <- assay(vsd)  
## "assay()" is part of the "SummarizedExperiment" package which is a DESeq2 dependency and is loaded with the DESeq2 library

Next, we need to compute the pairwise correlation values for all the samples. We can do this using the `cor()` function:

In [ ]:
### Compute pairwise correlation values
vsd_cor <- cor(vsd_mat)    ## cor() is a base R function

Let’s take a look at the column and row names of the correlation matrix.

In [ ]:
## check the output of cor(), make note of the row names and column names
head(vsd_cor)

collapsed_meta

You will notice that they match the names we have given our samples in the metadata data frame we started with. It is important that these match, so we can use the annotation argument below to plot a color block across the top. This block enables easy visualization of the hierarchical clustering.

Let’s plot the heatmap!

In [ ]:
### Plot heatmap using the correlation matrix and the metadata object
pheatmap(vsd_cor, annotation = root_meta)

Overall, the sample-to-sample correlation matrix shows very high correlations within biological replicates; for instance, CR1_5.bam, CR1_6.bam, and CR1_7.bam all have pairwise correlations above 0.996, and similar patterns are observed for the CR2 and CR3 sets. This consistency indicates strong reproducibility among replicates and no evidence of outlier samples.

As expected, samples from the same experimental group (e.g., Col-0 root replicates) cluster tightly together, while correlations are lower when comparing across distinct conditions such as genotype (Col-0 vs opt3-2) or tissue type (Root vs Shoot).

Taken together, these high correlation values and consistent clustering patterns confirm that the dataset is of high quality, providing confidence to proceed with differential expression analysis.

## 5. Differential expression analysis with DESeq2 <a id="5"></a>
The final step in the differential expression analysis workflow is fitting the raw counts to the NB model and performing the statistical test for differentially expressed genes. In this step we essentially want to determine whether the mean expression levels of different sample groups are significantly different.

![Differential_expression.png](images/M4P1_Intro_to_DE_Analysis_c5ad7c0a.png)

*Image: Paul Pavlidis (University of British Columbia), as reproduced in the HBC Intro-to-DGE lessons (CC BY 4.0).*

The [DESeq2 paper](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-014-0550-8) was published in 2014, but the package is continually updated and available for use in R through Bioconductor. It builds on good ideas for dispersion estimation and use of Generalized Linear Models from the DSS and edgeR methods.

Differential expression analysis with DESeq2 involves multiple steps as displayed in the flowchart below in blue. Briefly, DESeq2 will model the raw counts, using normalization factors (size factors) to account for differences in library depth. Then, it will estimate the gene-wise dispersions and shrink these estimates to generate more accurate estimates of dispersion to model the counts. Finally, DESeq2 will fit the negative binomial model and perform hypothesis testing using the Wald test or Likelihood Ratio Test.

![de_workflow_salmon_deseq1.png](images/M4P1_Intro_to_DE_Analysis_cef44e6d.png)


*NOTE: DESeq2 is actively maintained by the developers and continuously being updated. As such, it is important that you note the version you are working with. Recently, there have been some rather big changes implemented that impact the output. To find out more detail about the specific modifications made to methods described in the original 2014 paper, take a [look at this section in the DESeq2 vignette](http://bioconductor.org/packages/devel/bioc/vignettes/DESeq2/inst/doc/DESeq2.html#methods-changes-since-the-2014-deseq2-paper).*

### Running DESeq2

Prior to performing the differential expression analysis, it is a good idea to know what sources of variation are present in your data, either by exploration during the QC and/or prior knowledge. Once you know the major sources of variation, you can remove them prior to analysis or control for them in the statistical model by including them in your design formula.

**This step is critical as each additional factor in your design formula reduces your power. HOWEVER, failing to include important sources of variation can give you inaccurate results.**

#### Design formula

A design formula tells the statistical software which sources of variation to test for. This includes both your factor of interest as well as any additional covariates that are sources of variation. For example, if you know that sex is a significant source of variation in your data, then sex should be included in your model. The design formula should have all of the factors in your metadata that account for major sources of variation in your data.

For example, suppose you have the following metadata:
![meta_example.png](images/M4P1_Intro_to_DE_Analysis_1b7fea61.png)

if you want to examine the expression differences between treatments, and you know that major sources of variation include sex and age, then your design formula would be:

    design = ~ sex + age + treatment

The tilde (~) should always precede your factors and tells DESeq2 to model the counts using the following formula. Note the factors included in the design formula need to match the column names in the metadata.


*Does the order of variables matter?*

    In short, the order of variables in your design formula will not change the final results (i.e. the coefficients returned are the always the same). Typically, it has been best practice to list the variable that is your main effect in the last position of your design formula. In this way, the default result that is returned to you when using the results() function will be for your main effect.


## QUESTIONS: 

- Suppose you wanted to study the expression differences between the two age groups in the metadata shown above, and major sources of variation were sex and treatment, how would the design formula be written?
- Based on our GSE79275 metadata dataframe, which factors could we include in our design formula?
- What would you do if you wanted to include a factor in your design formula that is not in your metadata?

### GSE79275 DE analysis

Now that we know how to specify the model to DESeq2, we can run the differential expression pipeline on the raw counts.

To get our differential expression results from our raw count data, we only need to run 2 lines of code!

First we create a DESeqDataSet as we did in the ‘Count normalization’ lesson and specify the matrix which contains our raw counts, the metadata variable, and provide our design formula:

## Create DESeq2Dataset object
```r
dds <- DESeqDataSetFromMatrix(countData = data,
                              colData = meta,
                              design = ~ genotype * treatment)
```

Then, to run the actual differential expression analysis, we use a single call to the function DESeq().

## Run analysis
`dds <- DESeq(dds)`

By re-assigning the results of the function back to the same variable name (dds), we can fill in the slots of our DESeqDataSet object.

![deseq_obj2.png](images/M4P1_Intro_to_DE_Analysis_2d43a4da.png)

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = root_counts,
                              colData = root_meta,
                              design = ~ genotype + treatment + genotype:treatment)

In [ ]:
dds <- DESeq(dds)

### DESeq2 differential gene expression analysis workflow

We completed the entire workflow for the differential gene expression analysis with DESeq2. The steps in the analysis are output below:

![deseq2_workflow_separate.png](images/M4P1_Intro_to_DE_Analysis_05e8cc92.png)

We will be taking a detailed look at each of these steps to better understand how DESeq2 is performing the statistical analysis and what metrics we should examine to explore the quality of our analysis.

**Step 1: Estimate size factors**

The first step in the differential expression analysis is to estimate the size factors, which is exactly what we already did to normalize the raw counts.
![deseq2_workflow_separate_sf.png](images/M4P1_Intro_to_DE_Analysis_70dc77db.png)

DESeq2 will automatically estimate the size factors when performing the differential expression analysis. However, if you have already generated the size factors using estimateSizeFactors(), as we did earlier, then DESeq2 will use these values.

To normalize the count data, DESeq2 calculates size factors for each sample using the median of ratios method discussed previously in the ‘Count normalization’ lesson.
#### GSE79275 DE analysis: examining the size factors

Let’s take a quick look at size factor values we have for each sample:


In [ ]:
## Check the size factors
sizeFactors(dds)

These numbers should be identical to those we generated initially when we had run the function `estimateSizeFactors(dds)`. Take a look at the total number of reads for each sample:

In [ ]:
## Total number of raw counts per sample
colSums(counts(dds))

How do the numbers correlate with the size factor?

We see that the larger size factors correspond to the samples with higher sequencing depth, which makes sense, because to generate our normalized counts we need to divide the counts by the size factors. This accounts for the differences in sequencing depth between samples.

Now take a look at the total depth after normalization using:

In [ ]:
## Total number of normalized counts per sample
colSums(counts(dds, normalized=T))

    How do the values across samples compare with the total counts taken for each sample?

You might have expected the counts to be the exact same across the samples after normalization. However, DESeq2 also accounts for RNA composition during the normalization procedure. By using the median ratio value for the size factor, DESeq2 should not be biased to a large number of counts sucked up by a few DE genes; however, this may lead to the size factors being quite different than what would be anticipated just based on sequencing depth.

### Step 2: Estimate gene-wise dispersion

The next step in the differential expression analysis is the estimation of gene-wise dispersions. Before we get into the details, we should have a good idea about what dispersion refers to in DESeq2.
![deseq2_workflow_separate_dis.png](images/M4P1_Intro_to_DE_Analysis_b3140763.png)

In RNA-seq count data, we know:

1. To determine differentially expressed genes, we evaluate the variation of expression between groups (of interest) and compare that to the variation within the groups (between replicates).
2. For each individual gene, the mean is not equal to the variance.

- Genes that are highly expressed will have a more consistent level of variation, but it will be higher than the mean.
- Lowly expressed genes will exhibit variation that hovers around the mean (but with a higher amount of variability).

This complicated relationship means that we cannot just use the observed variance to account for within-group variation. Instead, DESeq2 uses dispersion.

![deseq_mean_variance2.png](images/M4P1_Intro_to_DE_Analysis_0fcdcfb2.png)

**What is dispersion?**

The dispersion parameter models the within-group variability by describing how much the variance deviates from the mean. A dispersion of 1 would indicate that there is no deviance from the mean (i.e, mean == variance). A typical RNA-seq dataset will exhibit some amount of biological variability present across replicates, and so we will always have dispersion values less than one.

![image.png](images/M4P1_Intro_to_DE_Analysis_d84c28dd.png)


DESeq2 estimates the dispersion for each gene based on the gene’s expression level (mean counts of within-group replicates) and observed variance across replicates, as we demonstrated with the formula above. In this way, the dispersion estimates for genes with the same mean will differ only based on their variance. **Therefore, the dispersion estimates reflect the variance in gene expression for a given mean value.**

![deseq_dispersion1.png](images/M4P1_Intro_to_DE_Analysis_c066e19e.png)

*See above: In this plot, we have dispersion on the y-axis and mean normalized counts on the x-axis. Each black dot represents a gene and its initial maximum likelihood dispersion estimate (MLE) given the observed data. Simply looking at the trend of black dots, we observe an inverse relationship between mean and dispersion. More details on the fitted red line and blue dots will be described later in this lesson.*

Since we have only a few (3-9) replicates per group, the dispersion estimates for each gene are often unreliable. As we walk through the next few steps, we will discuss how this issue is resolved.


**Step 3: Fit curve to gene-wise dispersion estimates**

The next step in the workflow is to fit a curve to the gene-wise dispersion estimates. The idea behind fitting a curve to the data is that different genes will have different scales of biological variability, but across all genes, there will be a distribution of reasonable estimates of dispersion.

![deseq2_workflow_separate_fit.png](images/M4P1_Intro_to_DE_Analysis_a257117c.png)

This curve is displayed as a red line in the figure presented below. This fitted line allows DESeq2 to utilize information across all genes to generate more accurate estimates using a method called ‘shrinkage’ (described in Step 4).

![deseq_dispersion1.png](images/M4P1_Intro_to_DE_Analysis_c066e19e.png)

**Step 4: Shrink gene-wise dispersion estimates toward the values predicted by the curve**
The next step in the workflow is to shrink the gene-wise dispersion estimates toward the expected dispersion values.

![deseq2_workflow_separate_shr.png](images/M4P1_Intro_to_DE_Analysis_9903e090.png)

DESeq2 assumes that genes with similar expression levels should have similar dispersion. As such, the fitted curve provides a range of expected dispersion values and a range of mean expression levels.

If the initial estimate (black dot) is much lower than the fitted curve, then values are shrunken (blue dots) towards the red line. Dispersion estimates that are slightly above the curve are also shrunk toward the curve for better dispersion estimation; however, genes with extremely high dispersion values are not (see right side figure below; these genes are shown surrounded by blue circle). This is due to the likelihood that the gene does not follow the modeling assumptions and has higher variability than others for biological or technical reasons.

![deseq_dispersion2.png](images/M4P1_Intro_to_DE_Analysis_f96c23f7.png)
*MAP, maximum a posteriori; MLE, maximum-likelihood estimate. For more detailed information, please see [Love MI, Huber W, Anders S, Genome Biology 2014](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-014-0550-8)*


The strength of the shrinkage for each gene depends on:

- how close gene dispersions are to the curve
- sample size (more samples = less shrinkage)

This dispersion shrinkage method is particularly important to reduce false positives in the differential expression analysis. This step allows for more accurate identification of differentially expressed genes when sample sizes are small.
Interpretation of the dispersion plot

To create this plot with your data, you use the function:

In [ ]:
plotDispEsts(dds)

This is a good plot to examine to ensure your data is a good fit for the DESeq2 model. Evaluate the plot to see if:

- The data generally scatters around the curve, with the dispersion decreasing with increasing mean expression levels.
    How much shrinkage do you get across the whole range of means in your data? For any experiment with low degrees of freedom, you will expect to see more shrinkage.
- The data scatter in a cloud or different shapes, then you might want to explore your data more to see if you have contamination (mitochondrial, etc.) or outlier samples.

**Examples of worrisome dispersion plots are shown below:**

The plot below shows a cloud of dispersion values, which do not generally follow the curve. This would be worrisome and suggests a bad fit of the data to the model.
![bad_dispersion1.png](images/M4P1_Intro_to_DE_Analysis_408704ac.png)

The next plot shows the dispersion values initially decreasing, then increasing with larger expression values. The larger mean expression values should not have larger dispersions based on our expectations - we expect decreasing dispersions with increasing mean. This indicates that there is less variation for more highly expressed genes than expected. This also indicates that there could be an outlier sample or contamination present in our analysis.

![bad_dispersion2.jpeg](images/M4P1_Intro_to_DE_Analysis_e5bc1c54.jpg)

In [ ]:
plotDispEsts(dds)

### GSE79275 DE analysis: exploring the dispersion estimates and assessing model fit

We see a nice decrease in dispersion with increasing mean expression, which is good. We also see the dispersion estimates generally surround the curve, which is also expected. Overall, this plot looks good.

## 6. Exploring Results (Wald test): <a id="6"></a>

By default, DESeq2 uses the Wald test to identify genes that are differentially expressed between two sample classes. Given the factor(s) used in the design formula, and how many factor levels are present, we can extract results for a number of different comparisons. Here, we will walk through how to obtain results from the `dds` object and provide some explanations on how to interpret them.

We are using a full factorial design, so `design = ~ genotype + treatment + genotype:treatment`

This means DESeq2 will model main effects of `genotype`, and `treatment`, as well as all interactions between them (`genotype:tissue:treatment`).

we can check all of these results with: 
`resultsNames(dds)`


In [ ]:
resultsNames(dds)

**IMPORTANT NOTE**: We will be exploring a specific set of conditions within the rest of the lab. For the exercise this week, you will investigate a different set of treatment conditions. 

For this interpretation, we are mainly interested in the effects of our treatment, exposure to cadmium for 72hrs, on the roots of the opt3 and col-0 plants. We want to see the interaction of this cadmium treatment across our opt3 mutants and across our col-0 controls. 

In our dataset, we are primarily interested in the effects of cadmium treatment (72 hrs) on the roots of Col-0 controls and opt3-2 mutants. To do this, we want to see both:

The treatment effect within Col-0 roots (Treated vs. Untreated).

The treatment effect within opt3-2 roots (Treated vs. Untreated).

Our results have these interactions:
| Coefficient                     | Meaning                | Comparison                                                                  |
|---------------------------------|------------------------|----------------------------------------------------------------------------|
| Intercept                        | Baseline expression    | Col-0 Untreated                                                            |
| genotype_opt3_vs_col0            | Main effect of genotype| opt3 vs Col-0 (Untreated)                                                 |
| treatment_Treated_vs_Untreated   | Main effect of treatment| Treated vs Untreated (Col-0)                                              |
| genotypeopt3.treatmentTreated    | Interaction effect     | (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)      |


Optionally, the interaction effect between genotype and treatment is used to determine if the cadmium response differs between opt3-2 and Col-0 roots.

When we initially created our `dds` object, we provided the design formula:
`design = ~ genotype + treatment + genotype:treatment`

This design captures the main effects of genotype and treatment, as well as all their interaction. To extract the comparisons of interest, we need to specify contrasts when using the results() function.

*NOTE: If we run the results() function without specifying contrast or name, it will return the comparison of the last level of the last variable in the design formula over the first level of this variable. If the order of levels is not specified, they are ordered alphabetically by DESeq2.*

Contrasts can be specified in two different ways (with the first method more commonly used):

- Contrasts can be supplied as a character vector with exactly three elements: the name of the factor (of interest) in the design formula, the name of the two factor levels to compare. The factor level given last is the base level for the comparison. The syntax is given below:
```
contrast <- c("factor_name", "level_of_interest", "base_level")
results(dds, contrast = contrast)
```

-Contrasts can be given as a list of 2 character vectors: the names of the fold changes for the level of interest, and the names of the fold changes for the base level. These names should match identically to the elements of resultsNames(object). This method can be useful for combining interaction terms and main effects.
```
resultsNames(dds)  # check available names
contrast_interaction <- list("treatmentTreated_vs_Untreated", "genotypeopt3-2.tissueRoot.treatmentTreated")
results(dds, contrast = contrast_interaction)
```
Alternatively, if you only had two factor levels you could do nothing and not worry about specifying contrasts (i.e. results(dds)). In this case, DESeq2 will choose what your base factor level based on alphabetical order of the levels.



    Does it matter what I choose to be my base level?

Yes, it does matter. Deciding what level is the base level will determine how to interpret the fold change that is reported. So for example, if we observe a log2 fold change of -2 this would mean the gene expression is lower in factor level of interest relative to the base level. Thus, if leaving it up to DESeq2 to decide on the contrasts be sure to check that the alphabetical order coincides with the fold change direction you are anticipating.


### The results table
Now that we have our contrast created, we can use it as input to the results() function. Let’s take a quick look at the help manual for the function:
`?results`

In [ ]:
results(dds, name = "genotypeopt3.treatmentTreated")

You will see we have the option to provide a wide array of arguments and tweak things from the defaults as needed. As we go through the lesson we will keep coming back to the help documentation to discuss some arguments that are good to know about.

In [ ]:
## Extract results for (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)
res_tableOE <- results(dds, name = "genotypeopt3.treatmentTreated")

In [ ]:
# What is stored in results?
res_tableOE %>% 
data.frame() %>% 
View()

We have six columns of information reported for each gene (row). We can use the mcols() function to extract information on what the values stored in each column represent:

In [ ]:
mcols(res_tableOE, use.names=T)

#### P-values

The p-value is a probability value used to determine whether there is evidence to reject the null hypothesis. A smaller p-value means that there is stronger evidence in favor of the alternative hypothesis. However, because we are performing a test for each individual gene we need to correct these p-values for multiple testing.

The padj column in the results table represents the p-value adjusted for multiple testing, and is the most important column of the results. Typically, a threshold such as padj < 0.05 is a good starting point for identifying significant genes. The default method for multiple test correction in DESeq2 is an implementation of the Benjamini-Hochberg false discovery rate (FDR). There are other correction methods available and can be changed by adding the pAdjustMethod argument to the results() function.

#### Gene-level filtering

Let’s take a closer look at our results table. As we scroll through it, you will notice that for selected genes,there are NA values in the pvalue and padj columns. What does this mean?

![gene_filtering.png](images/M4P1_Intro_to_DE_Analysis_81cbb197.png)

The missing values represent genes that have undergone filtering as part of the DESeq() function. Prior to differential expression analysis it is beneficial to omit genes that have little or no chance of being detected as differentially expressed. This will increase the power to detect differentially expressed genes. DESeq2 does not physically remove any genes from the original counts matrix, and so all genes will be present in your results table. The genes omitted by DESeq2 meet one of the three filtering criteria outlined below:

**1. Genes with zero counts in all samples**

If within a row, all samples have zero counts there is no expression information and therefore these genes are not tested.


In [ ]:
# Filter genes by zero expression
res_tableOE[which(res_tableOE$baseMean == 0),] %>% 
data.frame() %>% 
View()

#The baseMean column for these genes will be zero, and the log2 fold change estimates, p-value and adjusted p-value will all be set to NA.

**2. Genes with an extreme count outlier**

The `DESeq()` function calculates, for every gene and for every sample, a diagnostic test for outliers called Cook’s distance. Cook’s distance is a measure of how much a single sample influences the fitted coefficients for a gene, and a large value of Cook’s distance is intended to indicate an outlier count. Genes which contain a Cook’s distance above a threshold are flagged; however, at least 3 replicates are required for flagging, as it is difficult to judge which sample might be an outlier with only 2 replicates. We can turn off this filtering by using the cooksCutoff argument in the results() function.

In [ ]:
# Filter genes that have an extreme outlier
res_tableOE[which(is.na(res_tableOE$pvalue) & 
                    is.na(res_tableOE$padj) &
                    res_tableOE$baseMean > 0),] %>% 
  data.frame() %>% 
  View()

#If a gene contains a sample with an extreme count outlier then the p-value and adjusted p-value will be set to NA.

**3. Genes with a low mean normalized counts**

DESeq2 defines a low mean threshold, which is empirically determined from your data, in which the fraction of significant genes can be increased by reducing the number of genes that are considered for multiple testing. This is based on the notion that genes with very low counts are not likely to see a significant difference, typically due to high dispersion.

![indep_filt_scatterplus.png](images/M4P1_Intro_to_DE_Analysis_28251cb8.png)

At a user-specified value (`alpha = 0.05`), DESeq2 evaluates the change in the number of significant genes as it filters out increasingly larger portions of genes based on their mean counts, as shown in the figure above. The point at which the number of significant genes reaches a peak is the low mean threshold that is used to filter genes that undergo multiple testing. There is also an argument to turn off the filtering by setting `independentFiltering = F`.

In [ ]:
# Filter genes below the low mean threshold
res_tableOE[which(!is.na(res_tableOE$pvalue) & 
                    is.na(res_tableOE$padj) & 
                    res_tableOE$baseMean > 0),] %>% 
  data.frame() %>% 
  View()
#If a gene is filtered by independent filtering, then only the adjusted p-value will be set to NA.

#NOTE: DESeq2 will perform the filtering outlined above by default; however, other DE tools, such as EdgeR will not. Filtering is a necessary step, even if you are using limma-voom and/or edgeR’s quasi-likelihood methods. Be sure to follow pre-filtering steps when using other tools, as outlined in their user guides found on Bioconductor as they generally perform much better.


### Fold-Change
Another important column in the results table, is the log2FoldChange. With large significant gene lists it can be hard to extract meaningful biological relevance. To help increase stringency, one can also add a fold change threshold. Keep in mind when setting that value that we are working with log2 fold changes, so a cutoff of log2FoldChange < 1 would translate to an actual fold change of 2.

The fold changes reported in the results table are calculated by:
`log2 (normalized_counts_group1 / normalized_counts_group2)`

The problem is, these fold change estimates are not entirely accurate as they do not account for the large dispersion we observe with low read counts. To address this, the log2 fold changes need to be adjusted.

### More accurate LFC estimates

To generate more accurate log2 foldchange (LFC) estimates, DESeq2 allows for the shrinkage of the LFC estimates toward zero when the information for a gene is low, which could include:

    Low counts
    High dispersion values

LFC shrinkage uses information from all genes to generate more accurate estimates. Specifically, the distribution of LFC estimates for all genes is used (as a prior) to shrink the LFC estimates of genes with little information or high dispersion toward more likely (lower) LFC estimates.

![deseq2_shrunken_lfc.png](images/M4P1_Intro_to_DE_Analysis_d457a8ed.png)

*Image: Love, Huber & Anders (2014) Genome Biology 15:550, CC BY 4.0, [doi:10.1186/s13059-014-0550-8](https://doi.org/10.1186/s13059-014-0550-8).*

In the figure above, we have an example using two genes green gene and the purple gene. For each gene, the expression values are plotted for each sample in the two different mouse strains (C57BL/6J and DBA/2J). Both genes have the same mean values for the two sample groups, but the green gene has little variation within the group, while the purple gene has high levels of variation. For the green gene with low within-group variation, the unshrunken LFC estimate (vertex of the green solid line) is very similar to the shrunken LFC estimate (vertex of the green dotted line). However, LFC estimates for the purple gene are quite different due to the high dispersion. So even though two genes can have similar normalized count values, they can have differing degrees of LFC shrinkage. Notice the LFC estimates are shrunken toward the prior (black solid line).

Shrinking the log2 fold changes will not change the total number of genes that are identified as significantly differentially expressed. The shrinkage of the fold change is to help with the downstream assessment of results. For example, if you wanted to subset your significant genes based on fold change for further evaluation, you may want to use shrunken values. Additionally, for functional analysis tools such as GSEA, which require fold change values as input, you would want to provide shrunken values.

To generate the shrunken log2 fold change estimates, you have to run an additional step on your results object (that we will create below) with the function `lfcShrink()`.



In [ ]:
## Save the unshrunken results to compare
res_tableOE_unshrunken <- res_tableOE

# Apply fold change shrinkage
res_tableOE <- lfcShrink(dds, coef="genotypeopt3.treatmentTreated", type="apeglm")

Depending on the version of `DESeq2` you are using, the default method for shrinkage estimation will differ. The defaults can be changed by adding the argument type in the `lfcShrink()` function, as we have above. For the most recent versions of `DESeq2`, `type="normal"` is the default and was the only method in earlier versions. It has been shown that in most situations, there are alternative methods that have less bias than the ‘normal` method, and therefore, we chose to use apeglm.

Note that the stat column is no longer present in the results table. You can compare by inspecting the output of `res_tableOE` and `res_tableOE_unshrunken` in the R console. If you run `type="apeglm"` or `type="ashr"`, you do not get a stat column because these don’t really make sense in the context of the posterior quantities that are returned.


    How do I know what to value to provide to the coef argument?

The value you provide here needs to match identically what is stored in the column header of the coefficients table. To see what values you have to work with, you can use `resultsNames(dds)`.


### MA plot

A plot that can be useful for exploring our results is the MA plot. The MA plot shows the mean of the normalized counts versus the log2 fold changes for all genes tested. The genes that are significantly DE are colored to be easily identified. This is also a great way to illustrate the effect of LFC shrinkage. The DESeq2 package offers a simple function to generate an MA plot.

Let’s start with the unshrunken results:

In [ ]:
# MA plot using unshrunken fold changes
plotMA(res_tableOE_unshrunken, ylim=c(-2,2))

And now the shrunken results:

In [ ]:
# MA plot using shrunken fold changes
plotMA(res_tableOE, ylim=c(-2,2))

On the left you have the unshrunken fold change values plotted and you can see the abundance of scatter for the the lowly expressed genes. That is, many of the low expressors exhibit very high fold changes. After shrinkage, we see the fold changes are much smaller estimates.

In addition to the comparison described above, this plot allows us to evaluate the magnitude of fold changes and how they are distributed relative to mean expression. Generally, we would expect to see significant genes across the full range of expression levels.

## 7. Summarizing results <a id="7"></a>
To summarize the results table, a handy function in `DESeq2` is `summary()`. Confusingly, it has the same name as the function used to inspect data frames. This function, when called with a DESeq results table as input, will summarize the results using a default threshold of `padj < 0.1`. However, since we had set the alpha argument to 0.05 when creating our results table,e threshold: FDR < 0.05 (padj/FDR is used even though the output says `p-value < 0.05`). Let’s start with the (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated) results:

In [ ]:
## Summarize results
summary(res_tableOE, alpha = 0.05)

In addition to the number of genes up- and down-regulated at the default threshold, the function also reports the number of genes that were tested (genes with non-zero total read count), and the number of genes not included in multiple test correction due to a low mean count.

### Extracting significant differentially expressed genes

Let’s first create variables that contain our threshold criteria. We will only be using the adjusted p-values in our criteria:

In [ ]:
### Set thresholds
padj.cutoff <- 0.05

We can easily subset the results table to only include those that are significant using the `filter()` function, but first we will convert the results table into a tibble:

In [ ]:
# Create a tibble of results
res_tableOE_tb <- res_tableOE %>%
  data.frame() %>%
  rownames_to_column(var="gene") %>% 
  as_tibble()

In [ ]:
# Subset the tibble to keep only significant genes
sigOE <- res_tableOE_tb %>%
        dplyr::filter(padj < padj.cutoff)

In [ ]:
# Take a quick look at this tibble
sigOE

## 8. Visualizing the results<a id="8"></a>

We now have our list of DE genes from our results data frame for the `genotypeopt3.treatmentTreated = (opt3 Treated - opt3 Untreated) - (Col-0 Treated - Col-0 Untreated)`
comparison. What this will tell us is the interaction effect between our two combined treatments: 
- (opt3 Treated - opt3 Untreated): The effect of cadmium treatment on the opt3 mutants.
- (Col-0 Treated - Col-0 Untreated): The effect of cadmium treatment on the Col-0 controls.
- Interaction effect: The difference between these two treatment effects.

Interpretation:

- Positive log2 fold change → the gene responds more strongly to cadmium in opt3 roots than in Col-0 roots.

- Negative log2 fold change → the gene responds less strongly to cadmium in opt3 roots than in Col-0 roots.

- Near zero → the treatment effect is similar between genotypes.

When we are working with large amounts of data, it can be useful to display that information graphically to gain more insight. During this lesson, we will get you started with some basic and more advanced plots commonly used when exploring differential gene expression data; however, many of these plots can be helpful in visualizing other types of data as well.

We will be working with three different data objects that we have already created earlier :

1. Metadata for our samples (a dataframe): `root_meta`
2. Normalized expression data for every gene in each of our samples (a matrix): `normalized_counts`
3. Tibble versions of the DESeq2 results we generated: `res_tableOE_tb`

First, let’s create a metadata tibble from the data frame (don’t lose the row names!)

In [ ]:
roots_meta_names <- root_meta %>%
              rownames_to_column(var="samplename") %>% 
              as_tibble()

Next, let’s bring in a column with gene symbols to the normalized_counts object, so we can use them to label our plots. AGI are great for many things, but the gene symbols are much more recognizable to us, as biologists.

In [ ]:
library(biomaRt)
library(dplyr)

#Select the genes from sigOE
deg_genes <- sigOE$gene

#Connect to Ensembl Plants for Arabidopsis
mart <- useMart(
  biomart = "plants_mart",
  dataset = "athaliana_eg_gene",
  host = "https://plants.ensembl.org"
)

#Retrieve gene symbols
gene_info <- getBM(
  attributes = c("ensembl_gene_id", "external_gene_name"),
  filters = "ensembl_gene_id",
  values = deg_genes,
  mart = mart
)

#Create a mapping vector
gene_map <- setNames(gene_info$external_gene_name, gene_info$ensembl_gene_id)

#Add a 'common_name' column to sigOE
sigOE <- sigOE %>%
  mutate(common_name = gene_map[gene]) %>%
  mutate(common_name = ifelse(is.na(common_name), gene, common_name))  # keep original ID if no mapping

#Step 6: View the top rows
head(sigOE)

## Plotting significant DE genes

One way to visualize results would be to simply plot the expression data for a handful of genes. We could do that by picking out specific genes of interest or selecting a range of genes.

In [ ]:
# Plot expression for a single gene
plotCounts(dds, gene = "AT2G28160", intgroup = c("genotype", "treatment"))

In [ ]:
library(ggplot2)
fit_counts <- plotCounts(dds, gene = "AT2G28160", intgroup = c("genotype", "treatment"), returnData = TRUE)

ggplot(fit_counts, aes(x = treatment, y = count, color = genotype)) +
  geom_jitter(width = 0.2, size = 2) +
  geom_boxplot(aes(group = interaction(genotype, treatment)), alpha = 0.2, outlier.shape = NA) +
  scale_y_log10() +
  theme_bw() +
  labs(title = "FIT1 (AT2G28160) Counts by Genotype and Treatment")

### Heatmap

In addition to plotting subsets, we could also extract the normalized values of all the significant genes and plot a heatmap of their expression using `pheatmap()`.

In [ ]:
# Convert normalized_counts to a data frame with gene column
norm_counts_df <- as.data.frame(normalized_counts) %>%
  rownames_to_column(var = "gene")

# Subset metadata for roots of Col-0 and opt3
meta_subset <- root_meta %>%
  filter(tissue == "Root" & genotype %in% c("col0", "opt3")) %>%
  as.data.frame() %>%
  `rownames<-`(NULL) %>%
  column_to_rownames(var = "bio_id")

# Subset normalized counts for significant genes and matching root samples
norm_root_sig <- norm_counts_df %>%
  dplyr::filter(gene %in% sigOE$gene) %>%
  dplyr::select(gene, dplyr::all_of(as.character(rownames(meta_subset))))

# Convert to matrix for heatmap
mat <- as.matrix(norm_root_sig[, -1])
rownames(mat) <- norm_root_sig$gene



# Plot heatmap
pheatmap(mat,
         cluster_rows = TRUE,
         show_rownames = FALSE,
         annotation_col = meta_subset,
         border_color = NA,
         fontsize = 10,
         scale = "row",
         fontsize_row = 10,
         height = 20)

## Volcano plot

The above plot would be great to look at the expression levels of a good number of genes, but for more of a global view, there are other plots we can draw. A commonly used one is a volcano plot, in which you have the log-transformed adjusted p-values plotted on the y-axis and log2 fold change values on the x-axis.

To generate a volcano plot, we first need to have a column in our results data indicating whether or not the gene is considered differentially expressed based on p-adjusted values, and we will include a log2fold change here.

In [ ]:
## Obtain logical vector where TRUE values denote padj values < 0.05 and fold change > 1.5 in either direction

res_tableOE_tb <- res_tableOE_tb %>% 
                  dplyr::mutate(threshold_OE = padj < 0.05 & abs(log2FoldChange) >= 0.58)

## Volcano plot
ggplot(res_tableOE_tb) +
    geom_point(aes(x = log2FoldChange, y = -log10(padj), colour = threshold_OE)) +
    ggtitle("DE expression in genotypeopt3.treatmentTreated") +
    xlab("log2 fold change") + 
    ylab("-log10 adjusted p-value") +
    #scale_y_continuous(limits = c(0,50)) +
    theme(legend.position = "none",
          plot.title = element_text(size = rel(1.5), hjust = 0.5),
          axis.title = element_text(size = rel(1.25)))  

This is a great way to get an overall picture of what is going on, but what if we also wanted to know where the top 10 genes (lowest padj) in our DE list are located on this plot? We could label those dots with the gene name on the Volcano plot using geom_text_repel().

First, we need to order the `res_tableOE` tibble by `padj`, and add an additional column to it, to include on those gene names we want to use to label the plot.

In [ ]:
# --- Prepare labels for plotting ---
# Ensure res_tableOE_tb has a 'symbol' column; if not, create it from gene_map
if(!"symbol" %in% colnames(res_tableOE_tb)){
  res_tableOE_tb <- res_tableOE_tb %>%
    dplyr::mutate(symbol = gene_map[gene])
}

# Create empty column to store labels
res_tableOE_tb <- res_tableOE_tb %>% dplyr::mutate(genelabels = "")

# Sort by adjusted p-value
res_tableOE_tb <- res_tableOE_tb %>% dplyr::arrange(padj)

# Fill 'genelabels' with top 10 significant gene symbols
res_tableOE_tb$genelabels[1:10] <- as.character(res_tableOE_tb$symbol[1:10])

# View the table
View(res_tableOE_tb)

Next, we plot it as before with an additional layer for geom_text_repel() wherein we can specify the column of gene labels we just created.

In [ ]:
# Install ggrepel if you haven't already
if (!requireNamespace("ggrepel", quietly = TRUE)) {
  install.packages("ggrepel")
}

# Load the package
library(ggrepel)

ggplot(res_tableOE_tb, aes(x = log2FoldChange, y = -log10(padj))) +
    geom_point(aes(colour = threshold_OE)) +
    geom_text_repel(aes(label = genelabels)) +
    ggtitle("OPT3 and Col0 across cd treatments") +
    xlab("log2 fold change") + 
    ylab("-log10 adjusted p-value") +
    theme(legend.position = "none",
          plot.title = element_text(size = rel(1.5), hjust = 0.5),
          axis.title = element_text(size = rel(1.25))) 

Adapted from [HCB Training - Introduction to Differential Gene Expression Analysis](https://hbctraining.github.io/Intro-to-DGE/schedule/links-to-lessons.html)